# Tier 2 — N_train=5000 Experiments (Colab GPU)

Roda **18 modelos × 6 datasets × 30 seeds** com N_train=5000 (subsample estratificado por seed):
- **Baseline geral**: XGBoost
- **LSSVMs baselines (7)**: Std, PCP, FSA, IP, Pruning, OppM, FISTA-Nesterov
- **LSSVM paper-base**: ADMM-Nesterov (Marinho et al.)
- **LSSVM variantes/propostos (3)**: ADMM-ElasticNet, DualFISTA, Nyström-SVM
- **FT-Transformer baselines (4)**: Softmax, TopK, Entmax, Sparsemax
- **FT-Transformer inter-instâncias (2)**: SAINT (denso), FT-CUR (Nyströmformer)

Datasets: ADULT (45k), CREDIT (30k), BANK (45k), TELCO (7k), SHOPPERS (12k), HIGGS50K (50k).

**Antes de rodar:** `Runtime → Change runtime type → T4 GPU` (ou A100/V100 se Pro).

In [ ]:
# ── Célula 1: GPU check ─────────────────────────────────────────────────────
!nvidia-smi -L
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}')
    print(f'VRAM: {p.total_memory / 1e9:.1f} GB')
    print(f'CC: ({p.major}, {p.minor})')
else:
    print('⚠️ GPU não disponível — Runtime → Change runtime type → GPU')

In [ ]:
# ── Célula 2: Clonar do GitHub ──────────────────────────────────────────────
import os
PROJECT_DIR = '/content/sparse-lssvm-transformers-study'
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'

if os.path.exists(PROJECT_DIR):
    print(f'{PROJECT_DIR} já existe, fazendo pull...')
    !cd {PROJECT_DIR} && git pull --rebase
else:
    !git clone {GIT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'\nDiretório atual: {os.getcwd()}')

In [ ]:
# ── Célula 3: Instalar dependências ─────────────────────────────────────────
!pip install -q optuna entmax xgboost

import numpy, scipy, sklearn, torch, optuna, xgboost
print(f'numpy {numpy.__version__} | scipy {scipy.__version__} | '
      f'sklearn {sklearn.__version__} | torch {torch.__version__}')
print(f'optuna {optuna.__version__} | xgboost {xgboost.__version__}')

In [ ]:
# ── Célula 4: Baixar datasets (CREDIT, ADULT, BANK, TELCO, SHOPPERS, HIGGS50K) ─
!python scripts/download_data.py
print('\nDatasets disponíveis:')
!ls -lh data/raw/ | head -20

In [ ]:
# ── Célula 5: Montar Drive (para persistir resultados) ──────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao_tier2'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/tuning', exist_ok=True)
print(f'Drive em: {DRIVE_PATH}')
!ls -lh '{DRIVE_PATH}' 2>/dev/null

In [ ]:
# ── Célula 6: Restaurar progresso anterior (resume) ──────────────────────────
import shutil
from pathlib import Path

drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_results.mkdir(exist_ok=True)
(local_results / 'tuning').mkdir(exist_ok=True)

# Restaura resultados E params já tunados
for fname in ['tier2_n5000.json',
              'tuning/best_params_tier2_n5000.json']:
    src = drive_results / fname
    dst = local_results / fname
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'✓ Restaurado: {fname}')
    else:
        print(f'• Começando do zero: {fname}')

In [ ]:
# ── Célula 7: Sync helper (salva no Drive a cada N runs) ────────────────────
# Roda em background para fazer backup periódico
%%writefile /content/sync_to_drive.sh
#!/bin/bash
while true; do
    sleep 300  # 5 min
    cp -u /content/sparse-lssvm-transformers-study/results/tier2_n5000.json \
          "$1/tier2_n5000.json" 2>/dev/null
    cp -u /content/sparse-lssvm-transformers-study/results/tuning/best_params_tier2_n5000.json \
          "$1/tuning/best_params_tier2_n5000.json" 2>/dev/null
done

In [ ]:
# Inicia o sync em background
import subprocess
sync_proc = subprocess.Popen(['bash', '/content/sync_to_drive.sh', DRIVE_PATH])
print(f'Sync rodando em background (PID {sync_proc.pid}) — salva no Drive a cada 5 min')

In [ ]:
# ── Célula 8: Rodar experimentos completos ──────────────────────────────────
# 18 modelos × 6 datasets × 30 seeds = 3240 runs
# Tuning Optuna automático para os 14 modelos não-tunados (XGBoost+4 propostos já tunados)
# Em T4: ~10-15h. Em A100: ~5-8h.

!python scripts/run_tier2_n5000.py \
    --seeds 30 \
    --trials 20 \
    --folds 3 \
    --models-group all

In [ ]:
# ── Célula 9: Salvar resultados finais no Drive ─────────────────────────────
# Mata o sync background e copia versão final
import shutil, signal
from pathlib import Path

try:
    sync_proc.send_signal(signal.SIGTERM)
except Exception:
    pass

drive_dest = Path(DRIVE_PATH)
drive_dest.mkdir(parents=True, exist_ok=True)
(drive_dest / 'tuning').mkdir(exist_ok=True)

for fname in ['tier2_n5000.json',
              'tuning/best_params_tier2_n5000.json',
              'tier2_n5000.log']:
    src = Path('results') / fname
    dst = drive_dest / fname
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'✓ Salvo: {dst.name} ({src.stat().st_size / 1024:.1f} KB)')

print(f'\nConteúdo do Drive ({DRIVE_PATH}):')
!ls -lh '{DRIVE_PATH}'

In [ ]:
# ── Célula 10: Análise rápida (F1-macro por modelo × dataset) ─────────────
import json, numpy as np
from collections import defaultdict

results = json.load(open('results/tier2_n5000.json'))
print(f'Total runs: {len(results)}  ({sum(1 for r in results if r.get("status") == "ok")} OK)\n')

DATASETS = ['ADULT', 'CREDIT', 'BANK', 'TELCO', 'SHOPPERS', 'HIGGS50K']
scores = defaultdict(lambda: defaultdict(list))
for r in results:
    if r.get('status') != 'ok': continue
    m = r.get('model_variant') or r.get('model')
    scores[m][r['dataset']].append(r.get('f1_macro', float('nan')))

print(f'{"Modelo":<28}' + ''.join(f'{d:>10}' for d in DATASETS) + f'{"Média":>10}')
print('─' * 100)
rows = []
for m, ds_map in scores.items():
    vals = [np.mean(ds_map.get(d, [float("nan")])) for d in DATASETS]
    rows.append((m, vals, np.nanmean(vals)))
rows.sort(key=lambda x: -x[2])  # ordena por média
for m, vals, mean in rows:
    print(f'{m:<28}' + ''.join(f'{v:>10.4f}' if not np.isnan(v) else f'{"-":>10}'
                                for v in vals) + f'{mean:>10.4f}')